# 2026 Revenue Forecast (v2 + v3)

**Objective:** Monthly Jan–Dec 2026 revenue with three scenarios (Status Quo, Second-Purchase Push, Lazada Win-back).

**Data:** Gold parquets + teammate JSON (Roopa DS6, Benny BG/NBD, DS1/DS3 parameters).

**Models:**
- **v2** — seasonal acquisition baseline (repeat cohort + BG/NBD subscription)
- **v3** — regression acquisition (**recommended**) + Holt-Winters comparison

**Prerequisites:** `pandas`, `matplotlib`, `numpy`, `statsmodels`, `lifetimes` (Benny extractor).

Run cells top-to-bottom. Code families are grouped below; logic mirrors `scripts/build_forecast_2026.py`.


## Family A — Setup & configuration

Paths, imports, constants, and gold table loaders. Defines channel mapping and default forecast parameters merged with `configs/forecast_2026_params.json`.


In [ ]:
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

BASE = Path(".").resolve()
GOLD_DIR = BASE / "medallion" / "gold"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

FORECAST_VERSION = os.environ.get("FORECAST_VERSION", "v2")
PROMO_MONTHS = {3, 11, 12}

OVERALL_REPEAT_RATE = 0.2191
SUBSCRIBER_REPEAT_RATE = 0.85
NON_SUBSCRIBER_REPEAT_RATE = 0.195

DEFAULT_FORECAST_PARAMS = {
    "forecast_version": "v2",
    "promo_gap_pp": 0.0411,
    "promo_lost_revenue_per_cohort": 2626.0,
    "lazada_winback_conservative": 9776.0,
    "lazada_winback_upside": 43652.79,
    "annual_recovery_pivot": 5253.0,
    "sub_monthly_survival": 0.95,
    "promo_rate_reduction_factor": 0.7,
    "lazada_mix_shift_pp": 0.07,
    "seasonal_hist_start": "2024-01",
    "trend_cap_low": 0.85,
    "trend_cap_high": 1.15,
}

CHANNEL_MAP = {
    "DTC": "DTC", "Lazada": "Lazada", "Shopee": "Shopee",
    "Marketplace": "Other", "Other Marketplace": "Other", "Draft Order": "Other",
    "Bulk Import": "Other", "POS": "Other", "Email": "DTC", "TikTok": "Other",
    "Shop App": "Other", "Affiliate": "Other",
}
FORECAST_CHANNELS = ["DTC", "Lazada", "Shopee", "Other"]
DOCUMENTED_REPEAT = {"DTC": 0.2170, "Lazada": 0.2893, "Shopee": 0.1832, "Other": 0.1700}

print("Project root:", BASE)
print("Gold dir exists:", GOLD_DIR.exists())


## Family B — Teammate metric extractors

Replicates Roopa DS6 and Benny CLV notebook logic without rerunning their `.ipynb` files. Writes JSON snapshots consumed by the forecast engine.

Set `RUN_EXTRACTORS = False` to skip if JSON already exists in `outputs/`.


In [ ]:
RUN_EXTRACTORS = True

import json
from pathlib import Path

import pandas as pd


def load_first_orders_roopa_style(da: pd.DataFrame) -> pd.DataFrame:
    """First retail orders — same filters and promo_group rules as DS6 notebook."""
    first_orders = da[
        (da["is_first_order"] == True) & (da["is_b2b_or_affiliate"] == False)
    ].drop_duplicates(subset=["customer_id"])

    first_orders["promo_group"] = "Other/Small Discount"
    first_orders.loc[first_orders["discount_type"].isna(), "promo_group"] = "Full Price"
    first_orders.loc[first_orders["is_high_magnitude"] == True, "promo_group"] = "High Magnitude (30%+)"

    first_orders["repeat_purchase_90d"] = pd.to_numeric(
        first_orders["repeat_purchase_90d"], errors="coerce"
    )
    return first_orders


def extract_ds6_metrics() -> dict:
    base = BASE
    gold_dir = base / "medallion" / "gold"
    out_dir = base / "outputs"
    out_dir.mkdir(exist_ok=True)

    da = pd.read_parquet(gold_dir / "gold_discount_analysis.parquet")
    first_orders = load_first_orders_roopa_style(da)

    full_price_rate = float(
        first_orders.loc[first_orders["promo_group"] == "Full Price", "repeat_purchase_90d"].mean()
    )
    high_mag_rate = float(
        first_orders.loc[
            first_orders["promo_group"] == "High Magnitude (30%+)", "repeat_purchase_90d"
        ].mean()
    )
    gap = full_price_rate - high_mag_rate

    num_high_mag = int(
        first_orders.loc[
            first_orders["promo_group"] == "High Magnitude (30%+)", "customer_id"
        ].nunique()
    )

    # Roopa notebook uses FX-normalised price_total_sgd when present; else price_total.
    aov_col = "price_total_sgd" if "price_total_sgd" in first_orders.columns else "price_total"
    avg_order_val = float(first_orders[aov_col].mean())

    lost_repeat_customers = int(round(num_high_mag * gap))
    lost_revenue = float(lost_repeat_customers * avg_order_val)

    payload = {
        "promo_gap_pp": gap,
        "promo_lost_revenue_per_cohort": lost_revenue,
        "ds6_debug": {
            "full_price_rate": full_price_rate,
            "high_mag_rate": high_mag_rate,
            "n_high_mag_customers": num_high_mag,
            "avg_order_val": avg_order_val,
            "lost_repeat_customers": lost_repeat_customers,
            "source": "gold_discount_analysis.parquet (Roopa DS6 notebook alignment)",
        },
    }

    out_path = out_dir / "ds6_metrics.json"
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote {out_path}")
    return payload


import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


GOLD_DIR = BASE / "medallion" / "gold"
OUT_DIR = BASE / "outputs"

# Roopa notebook scenario assumptions (DS6 Part 4)
PIVOT_GAP_CLOSURE_PCT = 0.50
RISK_DISCOUNT_ACQ_RATE = 0.10
COHORT_CYCLES_PER_YEAR = 4


def load_first_orders_roopa_style(da: pd.DataFrame) -> pd.DataFrame:
    """First retail orders — same filters and promo_group rules as DS6 notebook."""
    first_orders = da[
        (da["is_first_order"] == True) & (da["is_b2b_or_affiliate"] == False)
    ].drop_duplicates(subset=["customer_id"])

    first_orders["promo_group"] = "Other/Small Discount"
    first_orders.loc[first_orders["discount_type"].isna(), "promo_group"] = "Full Price"
    first_orders.loc[first_orders["is_high_magnitude"] == True, "promo_group"] = "High Magnitude (30%+)"

    first_orders["repeat_purchase_90d"] = pd.to_numeric(
        first_orders["repeat_purchase_90d"], errors="coerce"
    )
    return first_orders


def _flag_holiday(ts: pd.Timestamp) -> str:
    if pd.isna(ts):
        return "Baseline"
    d = ts.to_pydatetime()
    if d.month == 11 and d.day == 11:
        return "11.11"
    if d.month == 12 and d.day == 12:
        return "12.12"
    if d.month == 11 and d.day >= 24:
        return "BFCM"
    return "Baseline"


def _build_scenario_simulation(first_orders: pd.DataFrame, gap: float, aov: float) -> dict:
    """
    Replicate Roopa DS6 Part 4 scenario simulation.

    Uses observed discount-acquisition rate on the full first-order base and
    Roopa's placeholder annual new-customer projection (13,715) for forward scenarios.
    """
    total_first_orders = int(first_orders["customer_id"].nunique())
    high_mag_customers = int(
        first_orders.loc[
            first_orders["promo_group"] == "High Magnitude (30%+)", "customer_id"
        ].nunique()
    )
    current_discount_rate = high_mag_customers / total_first_orders if total_first_orders else 0.0

    # Roopa notebook placeholder — conservative: same annual base as historical cohort
    projected_new_customers_2026 = total_first_orders

    sq_discount_cohort = int(round(projected_new_customers_2026 * current_discount_rate))
    sq_lost_repeats = int(round(sq_discount_cohort * gap))
    sq_lost_revenue = float(sq_lost_repeats * aov)
    sq_annual = float(sq_lost_revenue * COHORT_CYCLES_PER_YEAR)

    risk_discount_cohort = int(round(projected_new_customers_2026 * RISK_DISCOUNT_ACQ_RATE))
    risk_lost_repeats = int(round(risk_discount_cohort * gap))
    risk_lost_revenue = float(risk_lost_repeats * aov)
    risk_annual = float(risk_lost_revenue * COHORT_CYCLES_PER_YEAR)

    pivot_gap = gap * PIVOT_GAP_CLOSURE_PCT
    pivot_recovered_repeats = int(round(sq_discount_cohort * pivot_gap))
    pivot_recovered_revenue = float(pivot_recovered_repeats * aov)
    pivot_annual = float(pivot_recovered_revenue * COHORT_CYCLES_PER_YEAR)

    return {
        "aov": aov,
        "promo_gap_pp": gap,
        "projected_new_customers_2026": projected_new_customers_2026,
        "discount_acq_rate_status_quo": current_discount_rate,
        "discount_acq_rate_risk": RISK_DISCOUNT_ACQ_RATE,
        "pivot_gap_closure_pct": PIVOT_GAP_CLOSURE_PCT,
        "status_quo": {
            "discount_cohort_size": sq_discount_cohort,
            "lost_repeat_customers": sq_lost_repeats,
            "lost_revenue_per_cohort": sq_lost_revenue,
            "annual_leakage": sq_annual,
        },
        "risk": {
            "discount_cohort_size": risk_discount_cohort,
            "lost_repeat_customers": risk_lost_repeats,
            "lost_revenue_per_cohort": risk_lost_revenue,
            "annual_leakage": risk_annual,
        },
        "pivot": {
            "recovered_repeat_customers": pivot_recovered_repeats,
            "recovered_revenue_per_cohort": pivot_recovered_revenue,
            "annual_recovery": pivot_annual,
        },
        # Forecast wiring: spread Pivot recovery evenly across 2026 months.
        "annual_recovery_pivot": pivot_annual,
    }


def extract_ds6_roopa_metrics() -> dict:
    OUT_DIR.mkdir(exist_ok=True)

    da = pd.read_parquet(GOLD_DIR / "gold_discount_analysis.parquet")
    first_orders = load_first_orders_roopa_style(da)

    aov_col = "price_total_sgd" if "price_total_sgd" in first_orders.columns else "price_total"
    aov = float(first_orders[aov_col].mean())

    full_price_rate = float(
        first_orders.loc[first_orders["promo_group"] == "Full Price", "repeat_purchase_90d"].mean()
    )
    high_mag_rate = float(
        first_orders.loc[
            first_orders["promo_group"] == "High Magnitude (30%+)", "repeat_purchase_90d"
        ].mean()
    )
    gap = full_price_rate - high_mag_rate

    first_orders = first_orders.copy()
    first_orders["processed_at"] = pd.to_datetime(
        first_orders["processed_at"], errors="coerce", utc=True
    ).dt.tz_localize(None)
    first_orders["repeat_purchase_90d"] = pd.to_numeric(
        first_orders["repeat_purchase_90d"], errors="coerce"
    )
    first_orders["one_and_done"] = (
        pd.to_numeric(first_orders["total_orders"], errors="coerce").fillna(0) <= 1
    )
    first_orders["holiday_window"] = first_orders["processed_at"].apply(_flag_holiday)

    grp = (
        first_orders.groupby("holiday_window")
        .agg(
            n=("customer_id", "nunique"),
            one_and_done_rate=("one_and_done", "mean"),
            repeat_rate_90d=("repeat_purchase_90d", "mean"),
        )
        .fillna(0)
    )

    baseline = (
        grp.loc["Baseline"]
        if "Baseline" in grp.index
        else pd.Series({"n": 0, "one_and_done_rate": 0, "repeat_rate_90d": 0})
    )

    windows: dict[str, dict] = {}
    for w in ["11.11", "BFCM", "12.12"]:
        if w not in grp.index:
            continue
        row = grp.loc[w]
        repeat_vs_baseline = (
            float(row["repeat_rate_90d"] / baseline["repeat_rate_90d"])
            if baseline["repeat_rate_90d"]
            else 1.0
        )
        windows[w] = {
            "n": int(row["n"]),
            "one_and_done_rate": float(row["one_and_done_rate"]),
            "repeat_rate_90d": float(row["repeat_rate_90d"]),
            "repeat_vs_baseline": repeat_vs_baseline,
            "delta_one_and_done_pp": float((row["one_and_done_rate"] - baseline["one_and_done_rate"]) * 100),
            "delta_repeat_pp": float((row["repeat_rate_90d"] - baseline["repeat_rate_90d"]) * 100),
        }

    scenario_simulation = _build_scenario_simulation(first_orders, gap, aov)

    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
        "source_table": "gold_discount_analysis.parquet",
        "filters": {"is_first_order": True, "is_b2b_or_affiliate": False},
        "holiday_churn_trap": {
            "baseline": {
                "n": int(baseline["n"]),
                "one_and_done_rate": float(baseline["one_and_done_rate"]),
                "repeat_rate_90d": float(baseline["repeat_rate_90d"]),
            },
            "windows": windows,
            "forecast_month_map_2026": {"2026-11": ["11.11", "BFCM"], "2026-12": ["12.12"]},
        },
        "repeat_month_multipliers": {
            # Nov: BFCM multiplier (Roopa: more reliable sample than 11.11 alone).
            "2026-11": windows.get("BFCM", {}).get("repeat_vs_baseline", 1.0),
            "2026-12": windows.get("12.12", {}).get("repeat_vs_baseline", 1.0),
        },
        "scenario_simulation": scenario_simulation,
        "annual_recovery_pivot": scenario_simulation["annual_recovery_pivot"],
    }

    out_path = OUT_DIR / "ds6_roopa_metrics.json"
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote {out_path}")
    return payload


import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


def extract_benny_clv_metrics() -> dict:
    base = BASE
    gold_dir = base / "medallion" / "gold"
    out_dir = base / "outputs"
    out_dir.mkdir(exist_ok=True)

    # Lazy import so the error is obvious if missing.
    from lifetimes import BetaGeoFitter, GammaGammaFitter  # type: ignore

    horizons = [30, 60, 90, 180, 365]

    co = pd.read_parquet(gold_dir / "gold_customer_orders.parquet")
    cp = pd.read_parquet(gold_dir / "gold_customer_profiles.parquet")
    da = pd.read_parquet(gold_dir / "gold_discount_analysis.parquet")

    # Paid orders only.
    co_paid = co[co["payment_status"] == "paid"].copy()
    co_paid["processed_at"] = pd.to_datetime(co_paid["processed_at"], errors="coerce", utc=True)
    co_paid["processed_at_naive"] = co_paid["processed_at"].dt.tz_localize(None)

    # First paid order per customer.
    first_orders = (
        co_paid.sort_values(["customer_id", "processed_at_naive"])
        .groupby("customer_id", as_index=False)
        .first()[["customer_id", "processed_at_naive", "price_total", "channel", "utm_source", "utm_medium"]]
        .rename(columns={"processed_at_naive": "first_order_date", "price_total": "first_order_spend"})
    )

    obs_date = co_paid["processed_at_naive"].max()

    # Build RFM-like order rollups for all customers.
    agg = (
        co_paid.groupby("customer_id")
        .agg(
            total_orders=("order_id", "nunique"),
            first_order_date=("processed_at_naive", "min"),
            last_order_date=("processed_at_naive", "max"),
            total_spend=("price_total", "sum"),
        )
        .reset_index()
    )
    agg["AOF"] = (agg["total_orders"] - 1).clip(lower=0)
    agg["recency_days"] = (agg["last_order_date"] - agg["first_order_date"]).dt.days
    agg["tenure_days"] = (obs_date - agg["first_order_date"]).dt.days

    # Exclude B2B / affiliate (align to notebook scoping).
    b2b = set(da[da["is_b2b_or_affiliate"] == True]["customer_id"].dropna())
    rfm = agg[~agg["customer_id"].isin(b2b)].copy()

    # Remove zero tenure.
    rfm = rfm[rfm["tenure_days"] > 0].copy()

    # AOV proxy for Gamma-Gamma: repeat spend / AOF (repeat orders only).
    # First-order spend is approximated via the first paid order.
    rfm = rfm.merge(first_orders[["customer_id", "first_order_spend"]], on="customer_id", how="left")
    rfm["repeat_spend"] = rfm["total_spend"] - rfm["first_order_spend"].fillna(0)
    rfm["AOV_repeat"] = np.where(rfm["AOF"] > 0, rfm["repeat_spend"] / rfm["AOF"], 0.0)

    # Fit models.
    bgf = BetaGeoFitter(penalizer_coef=0.0)
    bgf.fit(rfm["AOF"], rfm["recency_days"], rfm["tenure_days"])

    gg_df = rfm[(rfm["AOF"] > 0) & (rfm["AOV_repeat"] > 0)].copy()
    ggf = GammaGammaFitter(penalizer_coef=0.0)
    ggf.fit(gg_df["AOF"], gg_df["AOV_repeat"])

    # Expected AOV (repeat order value) for all customers.
    rfm["exp_aov"] = ggf.conditional_expected_average_profit(rfm["AOF"], rfm["AOV_repeat"])

    portfolio = {}
    monthly_rates = {}
    for t in horizons:
        label = f"{t}d"
        exp_pur = bgf.conditional_expected_number_of_purchases_up_to_time(
            t, rfm["AOF"], rfm["recency_days"], rfm["tenure_days"]
        )
        exp_rev = (exp_pur * rfm["exp_aov"]).sum()
        monthly_rate = float(exp_rev / t * 30)
        portfolio[label] = {"exp_revenue_total": float(exp_rev), "monthly_rate": monthly_rate}
        monthly_rates[t] = monthly_rate

    # Proxy “survival” (steady-state ratio).
    sub_monthly_survival = float(monthly_rates[365] / monthly_rates[30]) if monthly_rates[30] else 0.95

    months_2026 = pd.period_range("2026-01", "2026-12", freq="M")
    subscription_forecast: dict[str, float] = {}
    rates = [float(portfolio.get(f"{h}d", {}).get("monthly_rate", 0.0)) for h in horizons]
    if any(rates):
        for i, m in enumerate(months_2026, start=1):
            days = min(i * 30, 365)
            subscription_forecast[str(m)] = float(np.interp(days, horizons, rates))

    payload = {
        "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "source_notebook_alignment": "04_clv_decay_forecast.ipynb (approximate filters)",
        "observation_date": obs_date.date().isoformat() if pd.notna(obs_date) else None,
        "model_population_n": int(len(rfm)),
        "bgnbd_params": {k: float(v) for k, v in bgf.params_.items()},
        "gamma_gamma_params": {k: float(v) for k, v in ggf.params_.items()},
        "portfolio": portfolio,
        "sub_monthly_survival_proxy": {
            "value": sub_monthly_survival,
            "formula": "monthly_rate_365d / monthly_rate_30d",
        },
        "subscription_forecast_monthly_2026": subscription_forecast,
    }

    out_path = out_dir / "clv_decay_metrics.json"
    out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Wrote {out_path}")
    return payload


if RUN_EXTRACTORS:
    extract_ds6_metrics()
    extract_ds6_roopa_metrics()
    extract_benny_clv_metrics()
    print("Extractor outputs refreshed in", OUTPUT_DIR)
else:
    print("Skipping extractors — using existing JSON in", OUTPUT_DIR)


## Family C — Parameter extraction

Merges config + JSON into `params`: repeat rates, AOVs, Lazada win-back, promo rates, subscription base.


In [ ]:
def _load_json_if_exists(path: Path) -> dict:
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def load_forecast_params() -> dict:
    cfg = _load_json_if_exists(BASE / "configs" / "forecast_2026_params.json")
    ds6 = _load_json_if_exists(OUTPUT_DIR / "ds6_metrics.json")
    roopa = _load_json_if_exists(OUTPUT_DIR / "ds6_roopa_metrics.json")
    clv = _load_json_if_exists(OUTPUT_DIR / "clv_decay_metrics.json")

    clv_override = {}
    if isinstance(clv.get("sub_monthly_survival_proxy"), dict) and "value" in clv["sub_monthly_survival_proxy"]:
        clv_override["sub_monthly_survival"] = clv["sub_monthly_survival_proxy"]["value"]

    roopa_override = {}
    if isinstance(roopa.get("repeat_month_multipliers"), dict):
        roopa_override["repeat_month_multipliers"] = roopa["repeat_month_multipliers"]
    sim = roopa.get("scenario_simulation") or {}
    if isinstance(sim, dict):
        if "annual_recovery_pivot" in sim:
            roopa_override["annual_recovery_pivot"] = sim["annual_recovery_pivot"]
        if "discount_acq_rate_status_quo" in sim:
            roopa_override["promo_acq_rate"] = sim["discount_acq_rate_status_quo"]
    elif "annual_recovery_pivot" in roopa:
        roopa_override["annual_recovery_pivot"] = roopa["annual_recovery_pivot"]

    if "subscription_forecast_monthly_2026" in clv:
        clv_override["subscription_forecast_monthly_2026"] = clv["subscription_forecast_monthly_2026"]

    return {**DEFAULT_FORECAST_PARAMS, **cfg, **ds6, **roopa_override, **clv_override}


def load_data():
    cp = pd.read_parquet(GOLD_DIR / "gold_customer_profiles.parquet")
    co = pd.read_parquet(GOLD_DIR / "gold_customer_orders.parquet")
    subs = pd.read_parquet(GOLD_DIR / "gold_subscription_behaviour.parquet")
    cohorts_ch = pd.read_parquet(GOLD_DIR / "gold_retention_cohorts_channel.parquet")
    return cp, co, subs, cohorts_ch


def prepare_customer_base(cp: pd.DataFrame, co: pd.DataFrame) -> pd.DataFrame:
    cp = cp.copy()
    cp["fc_channel"] = cp["acquisition_channel"].map(CHANNEL_MAP).fillna("Other")
    cp["first_month"] = pd.to_datetime(cp["first_order_date"]).dt.tz_localize(None).dt.to_period("M")
    first_orders = co[co["is_first_order"] == True][["customer_id", "price_total"]].rename(
        columns={"price_total": "first_order_aov"}
    )
    return cp.merge(first_orders, on="customer_id", how="left")


def extract_parameters(base: pd.DataFrame, subs: pd.DataFrame, forecast_params: dict | None = None) -> dict:
    channel_stats = (
        base.groupby("fc_channel")
        .agg(
            repeat_rate_90d=("repeat_purchase_90d", "mean"),
            first_order_aov=("first_order_aov", "mean"),
            customers=("customer_id", "count"),
        )
        .round(4)
    )
    repeat_rates, first_aov = {}, {}
    for ch in FORECAST_CHANNELS:
        if ch in channel_stats.index:
            repeat_rates[ch] = float(DOCUMENTED_REPEAT.get(ch, channel_stats.loc[ch, "repeat_rate_90d"]))
            first_aov[ch] = float(channel_stats.loc[ch, "first_order_aov"])
        else:
            repeat_rates[ch] = 0.17
            first_aov[ch] = 60.0

    fp = forecast_params or {}
    ds6_debug = fp.get("ds6_debug") or _load_json_if_exists(OUTPUT_DIR / "ds6_metrics.json").get("ds6_debug", {})
    full_price_rate = float(ds6_debug.get("full_price_rate", 0.2389))
    high_mag_rate = float(ds6_debug.get("high_mag_rate", 0.1978))
    promo_repeat_ratio = high_mag_rate / full_price_rate if full_price_rate else 0.83

    return {
        "overall_repeat_rate": OVERALL_REPEAT_RATE,
        "subscriber_repeat_rate": SUBSCRIBER_REPEAT_RATE,
        "non_subscriber_repeat_rate": NON_SUBSCRIBER_REPEAT_RATE,
        "repeat_rates": repeat_rates,
        "first_aov": first_aov,
        "active_subscribers": int((~subs["is_churned"].fillna(True)).sum()),
        "sub_aov": float(subs["avg_order_value"].mean()),
        "channel_stats": channel_stats,
        "promo_gap_pp": float(fp.get("promo_gap_pp", 0.0411)),
        "promo_lost_revenue_per_cohort": float(fp.get("promo_lost_revenue_per_cohort", 2626.0)),
        "lazada_winback_conservative": float(fp.get("lazada_winback_conservative", 9776.0)),
        "lazada_winback_upside": float(fp.get("lazada_winback_upside", 43652.79)),
        "sub_monthly_survival": float(fp.get("sub_monthly_survival", 0.95)),
        "annual_recovery_pivot": float(fp.get("annual_recovery_pivot", 5253.0)),
        "promo_acq_rate": float(fp.get("promo_acq_rate", 0.054)),
        "promo_repeat_ratio": promo_repeat_ratio,
        "promo_rate_reduction_factor": float(fp.get("promo_rate_reduction_factor", 0.7)),
        "lazada_mix_shift_pp": float(fp.get("lazada_mix_shift_pp", 0.07)),
        "forecast_version": fp.get("forecast_version", FORECAST_VERSION),
    }


def historical_monthly_acquisitions(base: pd.DataFrame, start: str = "2024-01", end: str | None = None) -> pd.DataFrame:
    hist = base[base["first_month"] >= start]
    if end is not None:
        hist = hist[hist["first_month"] <= end]
    hist = (
        hist.groupby(["first_month", "fc_channel"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=FORECAST_CHANNELS, fill_value=0)
    )
    return hist


## Family D — Revenue layer engine

`Monthly revenue = New acquisition + Repeat + Subscription (+ Win-back)`.

- **D1** Acquisition projections (v1 flat, v2 seasonal)
- **D2** Repeat cohort engine (Roopa promo split + holiday multipliers)
- **D3** Subscription (Benny BG/NBD curve) and scenario assembly


In [ ]:
# D1–D3: acquisition, repeat, subscription, assembly
# ---------------------------------------------------------------------------

def project_2026_monthly(hist: pd.DataFrame) -> pd.DataFrame:
    months_2026 = pd.period_range("2026-01", "2026-12", freq="M")
    recent = hist.loc[hist.index >= "2025-10"] if "2025-10" in hist.index else hist.tail(6)
    avg_monthly_total = recent.sum(axis=1).mean()
    anchor = hist.loc["2026-03"] if "2026-03" in hist.index else recent.iloc[-1]
    mix = anchor / anchor.sum()
    rows = []
    for m in months_2026:
        for ch in FORECAST_CHANNELS:
            rows.append({"month": m, "fc_channel": ch, "new_customers": avg_monthly_total * mix[ch]})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# v2 acquisition — seasonal + mild trend
# ---------------------------------------------------------------------------

def _seasonal_indices(hist: pd.DataFrame) -> dict[str, dict[int, float]]:
    seasonal: dict[str, dict[int, float]] = {}
    for ch in FORECAST_CHANNELS:
        series = hist[ch] if ch in hist.columns else pd.Series(0.0, index=hist.index)
        by_cal_month = series.groupby(series.index.month).mean()
        overall = float(series.mean()) if series.mean() > 0 else 1.0
        seasonal[ch] = {
            m: float(by_cal_month.get(m, overall) / overall) if overall else 1.0 for m in range(1, 13)
        }
    total = hist.sum(axis=1)
    by_total = total.groupby(total.index.month).mean()
    overall_t = float(total.mean()) if total.mean() > 0 else 1.0
    seasonal["_total"] = {
        m: float(by_total.get(m, overall_t) / overall_t) if overall_t else 1.0 for m in range(1, 13)
    }
    return seasonal


def project_2026_monthly_seasonal(
    hist: pd.DataFrame,
    forecast_params: dict,
    channel_mix: dict[str, float] | None = None,
) -> pd.DataFrame:
    hist_start = forecast_params.get("seasonal_hist_start", "2024-01")
    hist = hist.loc[hist.index >= hist_start] if len(hist) else hist
    months_2026 = pd.period_range("2026-01", "2026-12", freq="M")

    recent = hist.loc[hist.index >= "2025-10"] if "2025-10" in hist.index else hist.tail(6)
    base_run_rate = {ch: float(recent[ch].mean()) if ch in recent.columns else 0.0 for ch in FORECAST_CHANNELS}

    if len(hist) >= 6:
        recent3 = float(hist.tail(3).sum(axis=1).mean())
        prior3 = float(hist.iloc[-6:-3].sum(axis=1).mean())
        trend_factor = float(
            np.clip(recent3 / prior3 if prior3 else 1.0,
                    forecast_params.get("trend_cap_low", 0.85),
                    forecast_params.get("trend_cap_high", 1.15))
        )
    else:
        trend_factor = 1.0

    if channel_mix is None:
        anchor = hist.loc["2026-03"] if "2026-03" in hist.index else recent.sum()
        channel_mix = (anchor / anchor.sum()).to_dict()

    seasonal = _seasonal_indices(hist)
    rows = []
    for m in months_2026:
        cal_m = m.month
        total_seasonal = seasonal["_total"].get(cal_m, 1.0)
        month_total = sum(base_run_rate[ch] for ch in FORECAST_CHANNELS) * total_seasonal * trend_factor
        for ch in FORECAST_CHANNELS:
            ch_seasonal = seasonal[ch].get(cal_m, 1.0)
            raw = base_run_rate[ch] * ch_seasonal * trend_factor
            raw_sum = sum(base_run_rate[c] * seasonal[c].get(cal_m, 1.0) for c in FORECAST_CHANNELS) or 1.0
            mix_weight = channel_mix.get(ch, 0.25)
            n = month_total * mix_weight * (raw / raw_sum) if raw_sum else month_total * mix_weight
            rows.append({"month": m, "fc_channel": ch, "new_customers": n})
    return pd.DataFrame(rows)


def apply_lazada_mix_shift(projected: pd.DataFrame, shift_pp: float) -> pd.DataFrame:
    if shift_pp <= 0:
        return projected.copy()
    out = projected.copy()
    for month in out["month"].unique():
        mask = out["month"] == month
        sub = out.loc[mask].set_index("fc_channel")
        total = sub["new_customers"].sum()
        shift_n = total * shift_pp
        if "Shopee" in sub.index and "Lazada" in sub.index:
            sub.loc["Shopee", "new_customers"] = max(0.0, sub.loc["Shopee", "new_customers"] - shift_n)
            sub.loc["Lazada", "new_customers"] = sub.loc["Lazada", "new_customers"] + shift_n
        for ch in FORECAST_CHANNELS:
            out.loc[mask & (out["fc_channel"] == ch), "new_customers"] = sub.loc[ch, "new_customers"]
    return out


def build_new_acq_revenue(projected: pd.DataFrame, first_aov: dict) -> pd.DataFrame:
    projected = projected.copy()
    projected["new_acq_revenue"] = projected.apply(
        lambda r: r["new_customers"] * first_aov[r["fc_channel"]], axis=1
    )
    return projected.groupby("month").agg(
        new_customers=("new_customers", "sum"),
        new_acq_revenue=("new_acq_revenue", "sum"),
    )


# ---------------------------------------------------------------------------
# Repeat layers
# ---------------------------------------------------------------------------

def build_repeat_revenue(
    projected: pd.DataFrame,
    repeat_rates: dict,
    first_aov: dict,
    repeat_aov_factor: float = 0.95,
    repeat_month_multipliers: dict[str, float] | None = None,
) -> pd.Series:
    months = sorted(projected["month"].unique())
    proj_pivot = projected.pivot(index="month", columns="fc_channel", values="new_customers").fillna(0)
    repeat_by_month = {}
    for m in months:
        total_repeat = 0.0
        for lag in [1, 2, 3]:
            prior = m - lag
            if prior not in proj_pivot.index:
                continue
            for ch in FORECAST_CHANNELS:
                if ch not in proj_pivot.columns:
                    continue
                n = proj_pivot.loc[prior, ch]
                rate = repeat_rates[ch] / 3
                aov = first_aov[ch] * repeat_aov_factor
                total_repeat += n * rate * aov
        if repeat_month_multipliers:
            total_repeat *= float(repeat_month_multipliers.get(str(m), 1.0))
        repeat_by_month[m] = total_repeat
    return pd.Series(repeat_by_month, name="repeat_revenue")


def build_repeat_revenue_cohort(
    projected: pd.DataFrame,
    repeat_rates: dict,
    first_aov: dict,
    promo_acq_rate: float,
    promo_repeat_ratio: float,
    repeat_aov_factor: float = 0.95,
    repeat_month_multipliers: dict[str, float] | None = None,
) -> pd.Series:
    months = sorted(projected["month"].unique())
    proj_pivot = projected.pivot(index="month", columns="fc_channel", values="new_customers").fillna(0)
    repeat_by_month = {m: 0.0 for m in months}

    for t0 in months:
        for ch in FORECAST_CHANNELS:
            if ch not in proj_pivot.columns:
                continue
            n = float(proj_pivot.loc[t0, ch])
            n_promo = n * promo_acq_rate
            n_organic = n * (1.0 - promo_acq_rate)
            rate_org = repeat_rates[ch]
            rate_promo = repeat_rates[ch] * promo_repeat_ratio
            aov = first_aov[ch] * repeat_aov_factor
            for lag in [1, 2, 3]:
                t = t0 + lag
                if t not in repeat_by_month:
                    continue
                repeat_by_month[t] += (n_organic * rate_org / 3 + n_promo * rate_promo / 3) * aov

    if repeat_month_multipliers:
        for m in months:
            repeat_by_month[m] *= float(repeat_month_multipliers.get(str(m), 1.0))

    return pd.Series(repeat_by_month, name="repeat_revenue")


# ---------------------------------------------------------------------------
# Subscription layers
# ---------------------------------------------------------------------------

def build_subscription_revenue(params: dict, months: pd.PeriodIndex) -> pd.Series:
    active = params["active_subscribers"]
    sub_aov = params["sub_aov"]
    survival = params["sub_monthly_survival"]
    rev = {}
    current_active = active
    for m in months:
        rev[m] = current_active * sub_aov
        current_active *= survival
    return pd.Series(rev, name="subscription_revenue")


def build_subscription_revenue_v2(params: dict, months: pd.PeriodIndex, forecast_params: dict) -> pd.Series:
  monthly_json = forecast_params.get("subscription_forecast_monthly_2026")
  if isinstance(monthly_json, dict):
      rev = {}
      for m in months:
          rev[m] = float(monthly_json.get(str(m), monthly_json.get(m.strftime("%Y-%m"), 0.0)))
      if any(v > 0 for v in rev.values()):
          return pd.Series(rev, name="subscription_revenue")

  clv = _load_json_if_exists(OUTPUT_DIR / "clv_decay_metrics.json")
  portfolio = clv.get("portfolio", {})
  horizons = [30, 60, 90, 180, 365]
  rates = []
  for h in horizons:
      key = f"{h}d"
      rates.append(float(portfolio.get(key, {}).get("monthly_rate", 0.0)))
  if not any(rates):
      return build_subscription_revenue(params, months)

  base_jan = params["active_subscribers"] * params["sub_aov"]
  scale = base_jan / rates[0] if rates[0] else 1.0
  rev = {}
  for i, m in enumerate(months, start=1):
      days = min(i * 30, 365)
      rate = float(np.interp(days, horizons, rates))
      rev[m] = rate * scale
  return pd.Series(rev, name="subscription_revenue")


def assemble_scenario(
    months: pd.PeriodIndex,
    new_acq: pd.DataFrame,
    repeat_rev: pd.Series,
    sub_rev: pd.Series,
    scenario: str,
    repeat_multiplier: float = 1.0,
    repeat_monthly_addon: float = 0.0,
    winback_lump: float = 0.0,
    winback_month: str | None = None,
) -> pd.DataFrame:
    df = pd.DataFrame(index=months)
    df["scenario"] = scenario
    df["new_customers"] = new_acq["new_customers"]
    df["new_acq_revenue"] = new_acq["new_acq_revenue"]
    df["repeat_revenue"] = repeat_rev * repeat_multiplier + repeat_monthly_addon
    df["subscription_revenue"] = sub_rev
    df["winback_revenue"] = 0.0
    if winback_lump > 0 and winback_month:
        wm = pd.Period(winback_month, freq="M")
        if wm in df.index:
            df.loc[wm, "winback_revenue"] = winback_lump
    df["total_revenue"] = (
        df["new_acq_revenue"] + df["repeat_revenue"] + df["subscription_revenue"] + df["winback_revenue"]
    )
    return df.reset_index(names="month")


def historical_monthly_total_revenue(co: pd.DataFrame, start: str = "2024-01") -> pd.Series:
    orders = co.copy()
    orders["order_month"] = (
        pd.to_datetime(orders["processed_at"], utc=True, errors="coerce")
        .dt.tz_localize(None)
        .dt.to_period("M")
    )
    monthly = orders.dropna(subset=["order_month"]).groupby("order_month")["price_total"].sum().sort_index()
    return monthly.loc[monthly.index >= start]


## Family E — Acquisition forecasting (v3)

Regression (OLS: time + month + promo dummies) and Holt-Winters. Regression won Q1 2026 backtest on new customers (~9% MAPE).


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROMO_MONTHS = {3, 11, 12}

DEFAULT_ACQ_PARAMS_NB = {
    "hist_start": "2024-01",
    "end_train": "2025-12",
    "forecast_start": "2026-01",
    "forecast_end": "2026-12",
    "ewma_alpha": 0.35,
    "ewma_growth_window": 6,
    "mix_window": 3,
    "backtest_start": "2026-01",
    "backtest_end": "2026-03",
}


def load_acq_params() -> dict:
    cfg_path = BASE / "configs" / "forecast_2026_params.json"
    if cfg_path.exists():
        with cfg_path.open(encoding="utf-8") as f:
            cfg = json.load(f)
        acq = cfg.get("acquisition_forecast", {})
        return {**DEFAULT_ACQ_PARAMS_NB, **acq}
    return dict(DEFAULT_ACQ_PARAMS_NB)


def total_series(hist: pd.DataFrame) -> pd.Series:
    s = hist.sum(axis=1).astype(float)
    s.index = pd.PeriodIndex(s.index, freq="M")
    return s.sort_index()


def channel_mix(hist: pd.DataFrame, mix_window: int, channels: list[str]) -> pd.Series:
    recent = hist.tail(mix_window)
    mix = recent.sum()
    total = mix.sum()
    if total <= 0:
        mix = pd.Series(1.0 / len(channels), index=channels)
    else:
        mix = mix / total
    return mix.reindex(channels, fill_value=0.0)


def allocate_by_channel(
    total_forecast: pd.Series,
    hist: pd.DataFrame,
    mix_window: int,
    channels: list[str],
) -> pd.DataFrame:
    mix = channel_mix(hist, mix_window, channels)
    rows = []
    for month, total in total_forecast.items():
        for ch in channels:
            rows.append(
                {"month": month, "fc_channel": ch, "new_customers": float(total) * float(mix[ch])}
            )
    return pd.DataFrame(rows)


def forecast_regression(
    series: pd.Series,
    forecast_months: pd.PeriodIndex,
    end_train: pd.Period,
):
    train = series[series.index <= end_train]
    train_df = pd.DataFrame({"customers": train.values}, index=train.index)
    train_df["t"] = np.arange(len(train_df))
    train_df["cal_month"] = train_df.index.month
    train_df["promo"] = train_df["cal_month"].isin(PROMO_MONTHS).astype(int)
    if len(train_df) < 12:
        raise ValueError(f"Regression needs >=12 months; got {len(train_df)}")

    model = smf.ols("customers ~ t + C(cal_month) + promo", data=train_df).fit()
    last_t = int(train_df["t"].iloc[-1])
    pred_rows = []
    for i, m in enumerate(forecast_months, start=1):
        pred_rows.append(
            {"t": last_t + i, "cal_month": m.month, "promo": int(m.month in PROMO_MONTHS)}
        )
    pred_df = pd.DataFrame(pred_rows, index=forecast_months)
    pred = model.predict(pred_df).clip(lower=0.0)
    return pd.Series(pred.values, index=forecast_months, name="regression"), model


def forecast_holt_winters(
    series: pd.Series,
    forecast_months: pd.PeriodIndex,
    end_train: pd.Period,
) -> pd.Series:
    train = series[series.index <= end_train].astype(float)
    if len(train) < 24:
        raise ValueError(f"Holt-Winters needs >=24 months; got {len(train)}")

    best = None
    for trend in ("add", None):
        for seasonal in ("add", "mul"):
            try:
                model = ExponentialSmoothing(
                    train.values,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=12,
                    initialization_method="estimated",
                )
                fit = model.fit(optimized=True, use_brute=False)
                if best is None or fit.aic < best.aic:
                    best = fit
            except Exception:
                continue

    if best is None:
        model = ExponentialSmoothing(
            train.values,
            trend="add",
            seasonal="add",
            seasonal_periods=12,
            initialization_method="heuristic",
        )
        best = model.fit(optimized=True)

    fc = best.forecast(len(forecast_months))
    return pd.Series(fc, index=forecast_months, name="holt_winters")


def forecast_acquisition_total(
    method: str,
    hist_train: pd.DataFrame,
    months_2026: pd.PeriodIndex,
    end_train: pd.Period,
    ewma_alpha: float = 0.35,
    ewma_growth_window: int = 6,
) -> pd.Series:
    """Return monthly total new-customer forecast for the given method name."""
    total = total_series(hist_train)
    if method == "regression":
        fc, _ = forecast_regression(total, months_2026, end_train)
        return fc
    if method == "holt_winters":
        return forecast_holt_winters(total, months_2026, end_train)
    raise ValueError(f"Unknown acquisition method: {method}")


def project_acquisition_channels(
    method: str,
    hist_train: pd.DataFrame,
    months_2026: pd.PeriodIndex,
    channels: list[str],
    acq_params: dict | None = None,
) -> pd.DataFrame:
    """Forecast total new customers and split across channels."""
    acq_params = acq_params or load_acq_params()
    end_train = pd.Period(acq_params["end_train"], freq="M")
    total_fc = forecast_acquisition_total(
        method,
        hist_train,
        months_2026,
        end_train,
        ewma_alpha=acq_params.get("ewma_alpha", 0.35),
        ewma_growth_window=acq_params.get("ewma_growth_window", 6),
    )
    return allocate_by_channel(
        total_fc,
        hist_train,
        acq_params.get("mix_window", 3),
        channels,
    )


## Family F — Scenario builders, charts & validation

Assembles three scenarios on top of v2 or v3 acquisition. Includes plotting and validation helpers.


In [ ]:
# Scenario builders, charts, validation
def build_forecast(
    version: str | None = None,
    end_train: str | None = None,
    cp: pd.DataFrame | None = None,
    co: pd.DataFrame | None = None,
    subs: pd.DataFrame | None = None,
    cohorts_ch: pd.DataFrame | None = None,
    base: pd.DataFrame | None = None,
    write_outputs: bool = True,
):
    forecast_params = load_forecast_params()
    version = version or forecast_params.get("forecast_version", FORECAST_VERSION)

    if cp is None:
        cp, co, subs, cohorts_ch = load_data()
    if base is None:
        base = prepare_customer_base(cp, co)

    params = extract_parameters(base, subs, forecast_params)
    hist = historical_monthly_acquisitions(
        base, start=forecast_params.get("seasonal_hist_start", "2024-01"),
        end=end_train,
    )
    months_2026 = pd.period_range("2026-01", "2026-12", freq="M")
    repeat_mult = forecast_params.get("repeat_month_multipliers")

    if version == "v2":
        projected_sq = project_2026_monthly_seasonal(hist, forecast_params)
        projected_lazada = apply_lazada_mix_shift(projected_sq, params["lazada_mix_shift_pp"])
        promo_sq = params["promo_acq_rate"]
        promo_sp = params["promo_acq_rate"] * params["promo_rate_reduction_factor"]

        repeat_sq = build_repeat_revenue_cohort(
            projected_sq, params["repeat_rates"], params["first_aov"],
            promo_acq_rate=promo_sq, promo_repeat_ratio=params["promo_repeat_ratio"],
            repeat_month_multipliers=repeat_mult,
        )
        repeat_lazada = build_repeat_revenue_cohort(
            projected_lazada, params["repeat_rates"], params["first_aov"],
            promo_acq_rate=promo_sq, promo_repeat_ratio=params["promo_repeat_ratio"],
            repeat_month_multipliers=repeat_mult,
        )
        sub_rev = build_subscription_revenue_v2(params, months_2026, forecast_params)
        projected = projected_sq
    else:
        projected_sq = project_2026_monthly(hist)
        projected_lazada = projected_sq
        repeat_sq = build_repeat_revenue(
            projected_sq, params["repeat_rates"], params["first_aov"],
            repeat_month_multipliers=repeat_mult,
        )
        repeat_lazada = repeat_sq
        sub_rev = build_subscription_revenue(params, months_2026)
        projected = projected_sq

    new_acq_sq = build_new_acq_revenue(projected_sq, params["first_aov"])
    new_acq_lazada = build_new_acq_revenue(projected_lazada, params["first_aov"])

    status_quo = assemble_scenario(months_2026, new_acq_sq, repeat_sq, sub_rev, "Status Quo")
    pivot_monthly = params["annual_recovery_pivot"] / 12.0
    if version == "v2":
        promo_sp = params["promo_acq_rate"] * params["promo_rate_reduction_factor"]
        repeat_sp = build_repeat_revenue_cohort(
            projected_sq, params["repeat_rates"], params["first_aov"],
            promo_acq_rate=promo_sp,
            promo_repeat_ratio=params["promo_repeat_ratio"],
            repeat_month_multipliers=repeat_mult,
        )
    else:
        repeat_sp = repeat_sq

    second_purchase = assemble_scenario(
        months_2026, new_acq_sq, repeat_sp, sub_rev,
        "Second-Purchase Push", repeat_monthly_addon=pivot_monthly,
    )
    lazada_winback = assemble_scenario(
        months_2026, new_acq_lazada, repeat_lazada, sub_rev,
        "Lazada Win-back",
        winback_lump=params["lazada_winback_conservative"],
        winback_month="2026-03",
    )
    all_scenarios = pd.concat([status_quo, second_purchase, lazada_winback], ignore_index=True)

    if write_outputs:
        _write_all_outputs(all_scenarios, params, hist, co, projected, forecast_params, version, cohorts_ch)

    return all_scenarios, params, hist, projected


# ---------------------------------------------------------------------------
# v3 — regression / Holt-Winters acquisition + v2 revenue layers
# ---------------------------------------------------------------------------

def build_forecast_with_acquisition(
    acquisition_method: str,
    end_train: str | None = None,
    cp: pd.DataFrame | None = None,
    co: pd.DataFrame | None = None,
    subs: pd.DataFrame | None = None,
    cohorts_ch: pd.DataFrame | None = None,
    base: pd.DataFrame | None = None,
    write_outputs: bool = True,
):
    if acquisition_method not in ("regression", "holt_winters"):
        raise ValueError(f"acquisition_method must be regression or holt_winters, got {acquisition_method!r}")

    forecast_params = load_forecast_params()
    acq_params = load_acq_params()
    end_train = end_train or acq_params["end_train"]

    if cp is None:
        cp, co, subs, cohorts_ch = load_data()
    if base is None:
        base = prepare_customer_base(cp, co)

    params = extract_parameters(base, subs, forecast_params)
    hist = historical_monthly_acquisitions(
        base, start=acq_params.get("hist_start", "2024-01"), end=end_train,
    )
    months_2026 = pd.period_range("2026-01", "2026-12", freq="M")
    repeat_mult = forecast_params.get("repeat_month_multipliers")

    projected_sq = project_acquisition_channels(
        acquisition_method, hist, months_2026, FORECAST_CHANNELS, acq_params,
    )
    projected_lazada = apply_lazada_mix_shift(projected_sq, params["lazada_mix_shift_pp"])
    promo_sq = params["promo_acq_rate"]
    promo_sp = params["promo_acq_rate"] * params["promo_rate_reduction_factor"]

    repeat_sq = build_repeat_revenue_cohort(
        projected_sq, params["repeat_rates"], params["first_aov"],
        promo_acq_rate=promo_sq, promo_repeat_ratio=params["promo_repeat_ratio"],
        repeat_month_multipliers=repeat_mult,
    )
    repeat_lazada = build_repeat_revenue_cohort(
        projected_lazada, params["repeat_rates"], params["first_aov"],
        promo_acq_rate=promo_sq, promo_repeat_ratio=params["promo_repeat_ratio"],
        repeat_month_multipliers=repeat_mult,
    )
    repeat_sp = build_repeat_revenue_cohort(
        projected_sq, params["repeat_rates"], params["first_aov"],
        promo_acq_rate=promo_sp, promo_repeat_ratio=params["promo_repeat_ratio"],
        repeat_month_multipliers=repeat_mult,
    )
    sub_rev = build_subscription_revenue_v2(params, months_2026, forecast_params)

    new_acq_sq = build_new_acq_revenue(projected_sq, params["first_aov"])
    new_acq_lazada = build_new_acq_revenue(projected_lazada, params["first_aov"])

    status_quo = assemble_scenario(months_2026, new_acq_sq, repeat_sq, sub_rev, "Status Quo")
    pivot_monthly = params["annual_recovery_pivot"] / 12.0
    second_purchase = assemble_scenario(
        months_2026, new_acq_sq, repeat_sp, sub_rev,
        "Second-Purchase Push", repeat_monthly_addon=pivot_monthly,
    )
    lazada_winback = assemble_scenario(
        months_2026, new_acq_lazada, repeat_lazada, sub_rev,
        "Lazada Win-back",
        winback_lump=params["lazada_winback_conservative"],
        winback_month="2026-03",
    )
    all_scenarios = pd.concat([status_quo, second_purchase, lazada_winback], ignore_index=True)

    version_label = f"v3 ({acquisition_method})"
    if write_outputs:
        _write_scenario_outputs_for_method(
            all_scenarios, params, hist, co, projected_sq, acquisition_method,
            forecast_params, cohorts_ch, version_label,
        )

    return all_scenarios, params, hist, projected_sq


def _write_scenario_outputs_for_method(
    all_scenarios,
    params,
    hist,
    co,
    projected,
    acquisition_method,
    forecast_params,
    cohorts_ch,
    version_label,
):
    suffix = acquisition_method
    all_scenarios.to_csv(OUTPUT_DIR / f"forecast_2026_monthly_{suffix}.csv", index=False)
    sq = all_scenarios[all_scenarios["scenario"] == "Status Quo"]
    sq.to_csv(OUTPUT_DIR / f"forecast_2026_monthly_baseline_{suffix}.csv", index=False)
    projected.to_csv(OUTPUT_DIR / f"forecast_2026_acquisition_by_channel_{suffix}.csv", index=False)
    actual_revenue = historical_monthly_total_revenue(co, start="2024-01")
    plot_scenarios(
        all_scenarios,
        OUTPUT_DIR / f"forecast_2026_scenarios_{suffix}.png",
        actual_revenue=actual_revenue,
        version=version_label,
    )
    annual = plot_delta(all_scenarios, OUTPUT_DIR / f"forecast_2026_delta_{suffix}.png")
    plot_stacked_status_quo(sq, OUTPUT_DIR / f"forecast_2026_stacked_status_quo_{suffix}.png")
    plot_acquisition_seasonality(hist, projected, OUTPUT_DIR / f"forecast_2026_acquisition_seasonality_{suffix}.png")
    checks = run_validation(all_scenarios, params, projected, hist, cohorts_ch, version_label)
    return annual, checks


def run_acquisition_scenario_forecasts(
    methods: tuple[str, ...] = ("regression", "holt_winters"),
    end_train: str | None = None,
) -> dict:
    """Build 3-scenario revenue forecasts for each acquisition method; write separate charts."""
    cp, co, subs, cohorts_ch = load_data()
    base = prepare_customer_base(cp, co)
    results = {}
    summary_lines = [
        "# 2026 Revenue Scenarios by Acquisition Method",
        "",
        "Three scenarios (Status Quo, Second-Purchase Push, Lazada Win-back) with v2 repeat/subscription layers.",
        "",
        "| Acquisition method | Status Quo | Second-Purchase Push | Lazada Win-back | SP uplift | LZ uplift |",
        "|--------------------|------------|----------------------|-----------------|-----------|-----------|",
    ]

    for method in methods:
        all_scenarios, params, hist, projected = build_forecast_with_acquisition(
            method,
            end_train=end_train,
            cp=cp, co=co, subs=subs, cohorts_ch=cohorts_ch, base=base,
            write_outputs=True,
        )
        totals = all_scenarios.groupby("scenario")["total_revenue"].sum()
        sq, sp, lz = totals["Status Quo"], totals["Second-Purchase Push"], totals["Lazada Win-back"]
        summary_lines.append(
            f"| {method} | SGD {sq:,.0f} | SGD {sp:,.0f} | SGD {lz:,.0f} "
            f"| +{sp - sq:,.0f} | +{lz - sq:,.0f} |"
        )
        checks = run_validation(all_scenarios, params, projected, hist, cohorts_ch, f"v3-{method}")
        results[method] = {"scenarios": all_scenarios, "totals": totals, "checks": checks}
        print(f"\n=== {method} acquisition — 2026 scenario totals ===")
        for s, t in totals.items():
            print(f"  {s}: SGD {t:,.0f}")
        print(f"  Second-Purchase Push uplift: SGD {sp - sq:,.0f}")
        print(f"  Lazada Win-back uplift: SGD {lz - sq:,.0f}")

    summary_lines += [
        "",
        "## Outputs",
        "",
        "- `forecast_2026_scenarios_regression.png` — monthly revenue, 3 scenarios",
        "- `forecast_2026_scenarios_holt_winters.png` — monthly revenue, 3 scenarios",
        "- Matching delta and stacked charts per method",
    ]
    (OUTPUT_DIR / "forecast_scenarios_by_acquisition.md").write_text("\n".join(summary_lines), encoding="utf-8")
    return results


def _write_all_outputs(all_scenarios, params, hist, co, projected, forecast_params, version, cohorts_ch):
    all_scenarios.to_csv(OUTPUT_DIR / "forecast_2026_monthly.csv", index=False)
    sq = all_scenarios[all_scenarios["scenario"] == "Status Quo"]
    sq.to_csv(OUTPUT_DIR / "forecast_2026_monthly_baseline.csv", index=False)
    projected.to_csv(OUTPUT_DIR / "forecast_2026_acquisition_by_channel.csv", index=False)
    write_assumptions_md(params, hist, forecast_params, version, OUTPUT_DIR / "forecast_assumptions.md")
    actual_revenue = historical_monthly_total_revenue(co, start="2024-01")
    plot_scenarios(all_scenarios, OUTPUT_DIR / "forecast_2026_scenarios.png", actual_revenue=actual_revenue, version=version)
    plot_delta(all_scenarios, OUTPUT_DIR / "forecast_2026_delta.png")
    plot_stacked_status_quo(sq, OUTPUT_DIR / "forecast_2026_stacked_status_quo.png")
    plot_acquisition_seasonality(hist, projected, OUTPUT_DIR / "forecast_2026_acquisition_seasonality.png")
    checks = run_validation(all_scenarios, params, projected, hist, cohorts_ch, version)
    return checks


def write_assumptions_md(params: dict, hist: pd.DataFrame, forecast_params: dict, version: str, path: Path):
    lines = [
        "# 2026 Revenue Forecast Assumptions",
        "",
        f"**Forecast version:** `{version}`",
        "",
        "Auto-generated by `09_revenue_forecast_2026.ipynb` / `scripts/build_forecast_2026.py`.",
        "",
        "## Core parameters",
        "",
        "| Parameter | Value | Source |",
        "|-----------|-------|--------|",
        f"| Overall 90-day repeat rate | {params['overall_repeat_rate']:.2%} | DS1 |",
        f"| Promo retention gap | {params['promo_gap_pp']:.2%} | Roopa DS6 |",
        f"| Promo acquisition rate (status quo) | {params['promo_acq_rate']:.2%} | Roopa scenario_simulation |",
        f"| DS6 Pivot annual recovery | SGD {params.get('annual_recovery_pivot', 0):,.0f} | Roopa DS6 |",
        f"| Lazada win-back (conservative) | SGD {params['lazada_winback_conservative']:,.0f} | DS3-1 |",
        f"| Subscription monthly survival (v1 proxy) | {params['sub_monthly_survival']:.0%} | Benny BG/NBD proxy |",
        f"| Active subscribers (start) | {params['active_subscribers']} | gold_subscription_behaviour |",
        f"| Subscription AOV | SGD {params['sub_aov']:.2f} | gold_subscription_behaviour |",
        "",
        "## Channel repeat rates (90-day)",
        "",
        "| Channel | Rate | First-order AOV (SGD) |",
        "|---------|------|------------------------|",
    ]
    for ch in FORECAST_CHANNELS:
        lines.append(f"| {ch} | {params['repeat_rates'][ch]:.2%} | {params['first_aov'][ch]:.2f} |")

    anchor = hist.loc["2026-03"] if "2026-03" in hist.index else hist.iloc[-1]
    mix = (anchor / anchor.sum() * 100).round(1)
    lines += ["", "## 2026 acquisition mix anchor", "", "| Channel | Share |", "|---------|-------|"]
    for ch in FORECAST_CHANNELS:
        lines.append(f"| {ch} | {mix.get(ch, 0):.1f}% |")

    if version == "v2":
        lines += [
            "",
            "## v2 modeling choices",
            "",
            "- **Acquisition**: seasonal index by calendar month (from 2024+) × mild trend × Mar 2026 mix.",
            "- **Repeat**: cohort engine with organic vs promo split (Roopa rates); Nov/Dec holiday multipliers.",
            "- **Subscription**: BG/NBD portfolio monthly-rate interpolation (Benny `clv_decay_metrics.json`).",
            "- **Second-Purchase Push**: lower promo acquisition rate + Pivot recovery on repeat layer.",
            f"- **Lazada Win-back**: +{params['lazada_mix_shift_pp']:.0%} Lazada mix shift + Mar lump sum.",
            "",
            "## Roopa DS6 integration",
            "",
            "- `outputs/ds6_metrics.json` — promo gap",
            "- `outputs/ds6_roopa_metrics.json` — holiday + Pivot scenario",
        ]
    else:
        lines += [
            "",
            "## v1 modeling choices",
            "",
            "- Flat acquisition run-rate; simple repeat spread; subscription decay proxy.",
        ]
    path.write_text("\n".join(lines), encoding="utf-8")


def plot_scenarios(all_scenarios, output_path, actual_revenue=None, forecast_start="2026-01", version="v2"):
    fig, ax = plt.subplots(figsize=(14, 6))
    if actual_revenue is not None and len(actual_revenue) > 0:
        hist = actual_revenue.sort_index()
        ax.plot(hist.index.astype(str), hist.values, label="Actual revenue", color="#374151",
                linewidth=2.5, linestyle="-", marker="o", markersize=5, zorder=3)
    sq_total = all_scenarios[all_scenarios["scenario"] == "Status Quo"]["total_revenue"].sum()
    sp_total = all_scenarios[all_scenarios["scenario"] == "Second-Purchase Push"]["total_revenue"].sum()
    delta = sp_total - sq_total
    for scenario, color in [("Status Quo", "#6366f1"), ("Second-Purchase Push", "#22c55e"), ("Lazada Win-back", "#f59e0b")]:
        sub = all_scenarios[all_scenarios["scenario"] == scenario].sort_values("month")
        ax.plot(sub["month"].astype(str), sub["total_revenue"], label=f"{scenario} (forecast)",
                color=color, linewidth=2, linestyle="--", marker="o", markersize=5,
                markerfacecolor="white", markeredgewidth=1.5, markeredgecolor=color, zorder=2)
    ax.axvline(x=forecast_start, color="#9ca3af", linestyle=":", linewidth=1.5, alpha=0.9)
    ymax = ax.get_ylim()[1]
    ax.text(forecast_start, ymax * 0.97 if ymax > 0 else 1, "  Forecast →", va="top", ha="left", fontsize=9, color="#6b7280")
    ax.set_title(
        f"Monthly Revenue ({version}): Actuals vs 2026 Scenarios\nSecond-Purchase Push uplift vs Status Quo: SGD {delta:,.0f}",
        fontsize=13, fontweight="bold",
    )
    ax.set_xlabel("Month")
    ax.set_ylabel("Monthly Revenue (SGD)")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_delta(all_scenarios, output_path):
    annual = (
        all_scenarios.groupby("scenario")
        .agg(new_acq=("new_acq_revenue", "sum"), repeat=("repeat_revenue", "sum"),
             subscription=("subscription_revenue", "sum"), winback=("winback_revenue", "sum"),
             total=("total_revenue", "sum"))
        .loc[["Status Quo", "Second-Purchase Push", "Lazada Win-back"]]
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sq = annual.loc["Status Quo"]
    components = ["new_acq", "repeat", "subscription", "winback"]
    labels = ["New Acquisition", "Repeat", "Subscription", "Win-back"]
    colors = ["#6366f1", "#22c55e", "#a855f7", "#f59e0b"]
    axes[0].bar(labels, [sq[c] for c in components], color=colors)
    axes[0].set_title(f"Status Quo 2026 Revenue Breakdown\nTotal: SGD {sq['total']:,.0f}")
    axes[0].set_ylabel("SGD")
    for i, v in enumerate([sq[c] for c in components]):
        axes[0].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
    scenarios = annual.index.tolist()
    totals = annual["total"].values
    axes[1].bar(scenarios, totals, color=["#6366f1", "#22c55e", "#f59e0b"])
    axes[1].set_title("Annual 2026 Revenue by Scenario")
    axes[1].set_ylabel("SGD")
    for i, v in enumerate(totals):
        axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    return annual


def plot_stacked_status_quo(sq: pd.DataFrame, output_path: Path):
    fig, ax = plt.subplots(figsize=(12, 5))
    months = sq["month"].astype(str)
    bottom = np.zeros(len(sq))
    for col, label, color in [
        ("new_acq_revenue", "New Acquisition", "#6366f1"),
        ("repeat_revenue", "Repeat", "#22c55e"),
        ("subscription_revenue", "Subscription", "#a855f7"),
        ("winback_revenue", "Win-back", "#f59e0b"),
    ]:
        vals = sq[col].values
        ax.bar(months, vals, bottom=bottom, label=label, color=color)
        bottom += vals
    ax.set_title("Status Quo 2026 — Monthly Revenue by Layer")
    ax.set_ylabel("SGD")
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_acquisition_seasonality(hist: pd.DataFrame, projected: pd.DataFrame, output_path: Path):
    fig, ax = plt.subplots(figsize=(12, 5))
    hist_total = hist.sum(axis=1)
    ax.plot(hist_total.index.astype(str), hist_total.values, label="Actual new customers (monthly)", color="#374151", marker="o")
    proj_total = projected.groupby("month")["new_customers"].sum()
    ax.plot(proj_total.index.astype(str), proj_total.values, label="Forecast new customers (2026)", color="#6366f1", linestyle="--", marker="o")
    ax.set_title("New Customer Acquisition — History vs 2026 Forecast")
    ax.set_ylabel("New customers")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def run_validation(all_scenarios, params, projected, hist, cohorts_ch, version="v2"):
    checks = []
    sq = all_scenarios[all_scenarios["scenario"] == "Status Quo"]
    diff = (
        sq["total_revenue"] - sq["new_acq_revenue"] - sq["repeat_revenue"]
        - sq["subscription_revenue"] - sq["winback_revenue"]
    ).abs().max()
    checks.append(f"Component reconciliation max diff: SGD {diff:.2f} {'PASS' if diff < 1 else 'FAIL'}")

    if projected is not None:
        mix = projected.groupby("fc_channel")["new_customers"].sum()
        mix = mix / mix.sum()
        blended = sum(mix.get(ch, 0) * params["repeat_rates"][ch] for ch in FORECAST_CHANNELS)
        gap_pp = abs(blended - OVERALL_REPEAT_RATE) * 100
        status = "PASS" if gap_pp <= 1.0 else ("WARN (Shopee mix)" if gap_pp <= 2.0 else "FAIL")
        checks.append(f"Blended repeat rate {blended:.2%} vs DS1 {OVERALL_REPEAT_RATE:.2%} (gap {gap_pp:.2f}pp): {status}")

    if version == "v2" and projected is not None:
        proj_monthly = projected.groupby("month")["new_customers"].sum()
        nov = float(proj_monthly.get(pd.Period("2026-11", freq="M"), 0))
        feb = float(proj_monthly.get(pd.Period("2026-02", freq="M"), 0))
        checks.append(f"Nov new customers > Feb: {'PASS' if nov > feb else 'WARN'} ({nov:.0f} vs {feb:.0f})")

    totals = all_scenarios.groupby("scenario")["total_revenue"].sum()
    checks.append(f"Second-Purchase Push >= Status Quo: {'PASS' if totals['Second-Purchase Push'] >= totals['Status Quo'] else 'FAIL'}")
    checks.append(f"Lazada Win-back >= Status Quo: {'PASS' if totals['Lazada Win-back'] >= totals['Status Quo'] else 'FAIL'}")
    return checks


## Family G — Revenue backtest (v1 vs v2)

Trains through Dec 2025; evaluates Q1 2026 actual revenue vs Status Quo forecast.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _metrics(actual: pd.Series, forecast: pd.Series) -> dict:
    aligned = pd.concat([actual.rename("actual"), forecast.rename("forecast")], axis=1).dropna()
    if aligned.empty:
        return {"mape": np.nan, "mae": np.nan, "bias": np.nan, "n_months": 0}
    err = aligned["forecast"] - aligned["actual"]
    mape = (err.abs() / aligned["actual"].replace(0, np.nan)).mean() * 100
    return {
        "mape": float(mape),
        "mae": float(err.abs().mean()),
        "bias": float(err.mean()),
        "n_months": int(len(aligned)),
    }


def run_backtest(end_train: str = "2025-12", eval_start: str = "2026-01", eval_end: str = "2026-03") -> pd.DataFrame:
    cp, co, subs, cohorts_ch = load_data()
    base = prepare_customer_base(cp, co)
    actual = historical_monthly_total_revenue(co, start=eval_start)
    actual = actual[(actual.index >= eval_start) & (actual.index <= eval_end)]

    rows = []
    for version in ("v1", "v2"):
        all_scenarios, _, _, _ = build_forecast(
            version=version,
            end_train=end_train,
            cp=cp,
            co=co,
            subs=subs,
            cohorts_ch=cohorts_ch,
            base=base,
            write_outputs=False,
        )
        sq = all_scenarios[all_scenarios["scenario"] == "Status Quo"].copy()
        sq["month"] = pd.PeriodIndex(sq["month"], freq="M")
        fc = sq.set_index("month")["total_revenue"]
        fc = fc[(fc.index >= eval_start) & (fc.index <= eval_end)]
        m = _metrics(actual, fc)
        for month in fc.index:
            rows.append(
                {
                    "version": version,
                    "month": str(month),
                    "actual": float(actual.get(month, np.nan)),
                    "forecast": float(fc.get(month, np.nan)),
                    "error": float(fc.get(month, np.nan) - actual.get(month, np.nan)),
                }
            )
        rows.append({"version": version, "month": "SUMMARY", **m})

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "forecast_backtest_comparison.csv", index=False)

    lines = [
        "# Forecast backtest (Status Quo vs actuals)",
        "",
        f"Train acquisitions through **{end_train}**. Evaluate **{eval_start}** to **{eval_end}**.",
        "",
        "| Version | MAPE | MAE (SGD) | Bias (SGD) |",
        "|---------|------|-----------|------------|",
    ]
    for version in ("v1", "v2"):
        s = df[(df["version"] == version) & (df["month"] == "SUMMARY")].iloc[0]
        lines.append(
            f"| {version} | {s['mape']:.1f}% | {s['mae']:,.0f} | {s['bias']:+,.0f} |"
        )
    (OUTPUT_DIR / "forecast_backtest_summary.md").write_text("\n".join(lines), encoding="utf-8")
    return df


## Section 10 — v2 baseline (optional)

Seasonal acquisition model kept for comparison.


In [ ]:
cp, co, subs, cohorts_ch = load_data()
base = prepare_customer_base(cp, co)

all_scenarios_v2, params_v2, hist_v2, projected_v2 = build_forecast(
    version="v2",
    cp=cp, co=co, subs=subs, cohorts_ch=cohorts_ch, base=base,
    write_outputs=True,
)
totals_v2 = all_scenarios_v2.groupby("scenario")["total_revenue"].sum()
print("=== v2 annual totals ===")
print(totals_v2.to_string())


## Section 11 — Acquisition method comparison

Compare regression, Holt-Winters, and v2 seasonal on **new customers** (Q1 2026).


In [ ]:
acq_params = load_acq_params()
end_train_p = pd.Period(acq_params["end_train"], freq="M")
forecast_months = pd.period_range("2026-01", "2026-12", freq="M")
backtest_months = pd.period_range(acq_params["backtest_start"], acq_params["backtest_end"], freq="M")

hist_train = historical_monthly_acquisitions(base, start=acq_params["hist_start"], end=acq_params["end_train"])
hist_full = historical_monthly_acquisitions(base, start=acq_params["hist_start"])
total_train = total_series(hist_train)
actual_bt = total_series(hist_full).reindex(backtest_months)

methods = {
    "regression": forecast_regression(total_train, forecast_months, end_train_p)[0],
    "holt_winters": forecast_holt_winters(total_train, forecast_months, end_train_p),
    "v2_seasonal": project_2026_monthly_seasonal(hist_train, load_forecast_params()).groupby("month")["new_customers"].sum(),
}

rows = []
for name, fc in methods.items():
    aligned = pd.concat([actual_bt.rename("actual"), fc.reindex(backtest_months).rename("forecast")], axis=1).dropna()
    mape = ((aligned["forecast"] - aligned["actual"]).abs() / aligned["actual"]).mean() * 100
    rows.append({"method": name, "Q1_MAPE_pct": round(mape, 1)})
pd.DataFrame(rows).sort_values("Q1_MAPE_pct")


## Section 12 — v3 scenarios (main result)

**Regression** (recommended) and **Holt-Winters** acquisition with three revenue scenarios each.


In [ ]:
v3_results = run_acquisition_scenario_forecasts(methods=("regression", "holt_winters"))

summary_path = OUTPUT_DIR / "forecast_scenarios_by_acquisition.md"
if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))


## Section 13 — Charts, assumptions & slide summary


In [ ]:
from IPython.display import Image, display

for name in [
    "forecast_2026_scenarios_regression.png",
    "forecast_2026_scenarios_holt_winters.png",
    "forecast_2026_delta_regression.png",
    "acquisition_forecast_comparison.png",
]:
    p = OUTPUT_DIR / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

assumptions_path = OUTPUT_DIR / "forecast_assumptions.md"
if assumptions_path.exists():
    print("\n--- Assumptions (excerpt) ---")
    print(assumptions_path.read_text(encoding="utf-8")[:900])

reg = v3_results["regression"]["totals"]
sq, sp, lz = reg["Status Quo"], reg["Second-Purchase Push"], reg["Lazada Win-back"]
print(f"\n=== Slide bullets (regression acquisition) ===")
print(f"Status Quo 2026: SGD {sq:,.0f}")
print(f"Second-Purchase Push: SGD {sp:,.0f} (+SGD {sp - sq:,.0f})")
print(f"Lazada Win-back: SGD {lz:,.0f} (+SGD {lz - sq:,.0f})")

backtest_df = run_backtest()
print("\n=== Revenue backtest (v1 vs v2 Status Quo) ===")
print(backtest_df[backtest_df["month"] == "SUMMARY"].to_string(index=False))
